# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# ## Method choice and why
# (asked claude to ask me questions, i answered and then asked it to format and refine the answers. these answers are pasted below)
# We use two methods: **Decision Tree** and **Random Forest**, both classification methods
# (our target, `is_declining`, is a 0/1 label — not a continuous number, so Linear Regression
# is excluded; Logistic Regression, despite its name, IS a valid classification method, but we
# chose tree-based methods for the specific reason below).

# **Decision Tree** is chosen for its transparency: it can be printed as literal if/else rules,
# which matters directly for our lane — reviewers using our ranked queue need to trust *why* a
# page was flagged (same principle behind our w04 reason codes). A model whose logic can be
# read and verified builds more trust than an opaque score.

# **Random Forest** is included as a comparison, since it typically performs better (more
# predictive power, per notebook 02's reference numbers) but sacrifices that direct readability
# — it averages many trees together rather than offering one traceable path. Comparing both
# lets us see the real tradeoff between interpretability and performance, and choose deliberately
# rather than assuming "more complex = better."

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# ## Split design

# We use a **grouped split by `client_hash_id`**, not a plain random split. Recall from w03's
# lane guide: "pages from the same client may share patterns the model could memorize" — if the
# same client's pages appeared in both train and test, the model could learn client-specific
# quirks (a particular site's style, industry norms) rather than general decline patterns,
# inflating our test score without real generalization. This matches the starter pipeline's own
# approach (`split_strategy: client_holdout`, from notebook 01).

# We use `GroupShuffleSplit` from scikit-learn, grouping on `client_hash_id`, so entire clients
# are held out for testing — never split across train and test.

We considered a time-aware split (train on earlier periods, test on later ones) but determined
it wasn't necessary here: our feature window (February) and label window (March) are already
fixed and non-overlapping by design (verified in w03/w04), so there's no rolling-time risk to
guard against. The client-holdout split addresses the more relevant risk for this setup —
preventing the model from memorizing client-specific patterns rather than learning genuine
decline signals.

In [3]:
# --- 1. Token + connection ---
import os
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


In [4]:
# --- 2. February features (now including client_hash_id) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions_ctr,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

content_features = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume
    FROM {TABLES['dim_content']}
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)
print("content_features:", content_features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_impressions: (153559, 3)
feb_ctr: (153559, 4)
feb_position: (151956, 2)
content_features: (519606, 3)


In [5]:
# --- 3. March impressions + label ---
march_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

trend_data = feb_impressions.merge(march_impressions, on="content_hash_id", how="inner")
trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print("trend_data:", trend_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trend_data: (134238, 5)


In [6]:
# --- 4. Assemble final training table WITH client_hash_id ---
training_table = content_features \
    .merge(feb_impressions, on="content_hash_id", how="inner") \
    .merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="left") \
    .merge(feb_position, on="content_hash_id", how="left") \
    .merge(trend_data[["content_hash_id", "is_declining"]], on="content_hash_id", how="inner")

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
training_table_clean = training_table.dropna(subset=feature_cols)

print("training_table_clean:", training_table_clean.shape)
print(training_table_clean.columns.tolist())

training_table_clean: (75189, 8)
['content_hash_id', 'word_count', 'search_volume', 'client_hash_id', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'is_declining']


In [7]:
from sklearn.model_selection import GroupShuffleSplit

# We need client_hash_id in our training table to group by it.
# Let's check it's still there from your feature-building steps.
print(training_table_clean.columns.tolist())

['content_hash_id', 'word_count', 'search_volume', 'client_hash_id', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'is_declining']


In [8]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
X = training_table_clean[feature_cols]
y = training_table_clean["is_declining"]
groups = training_table_clean["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Verify: no client appears in both sets
train_clients = set(training_table_clean.iloc[train_idx]["client_hash_id"])
test_clients = set(training_table_clean.iloc[test_idx]["client_hash_id"])
overlap = train_clients & test_clients

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train clients:", len(train_clients), "| Test clients:", len(test_clients))
print("Overlapping clients (should be 0):", len(overlap))

Train rows: 52791 | Test rows: 22398
Train clients: 28 | Test clients: 10
Overlapping clients (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
# Rebuild the baseline score (from w04) on the TEST set only, for fair comparison
test_table = training_table_clean.iloc[test_idx].copy()

# Same rule as w04: good position (<=10) AND low CTR (<0.006083) -> qualifies
good_position = (test_table["feb_avg_position"] <= 10).astype(int)
low_ctr = (test_table["feb_ctr"] < 0.006083).astype(int)
test_table["baseline_score"] = good_position * low_ctr * test_table["feb_impressions"]

print("Baseline qualifying pages in test set:", (test_table["baseline_score"] > 0).sum())

Baseline qualifying pages in test set: 10869


In [10]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y_test_arr = test_table["is_declining"].values

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
forest.fit(X_train, y_train)

tree_scores = tree.predict_proba(X_test)[:, 1]
forest_scores = forest.predict_proba(X_test)[:, 1]

print("Models trained.")

Models trained.


In [12]:
results = []
for k in (20, 50):
    results.append({
        "method": "baseline_rule",
        "k": k,
        "precision": precision_at_k(test_table["baseline_score"].values, y_test_arr, k)
    })
    results.append({
        "method": "decision_tree",
        "k": k,
        "precision": precision_at_k(tree_scores, y_test_arr, k)
    })
    results.append({
        "method": "random_forest",
        "k": k,
        "precision": precision_at_k(forest_scores, y_test_arr, k)
    })

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df.pivot(index="method", columns="k", values="precision"))

k                20    50
method                   
baseline_rule  0.15  0.22
decision_tree  0.05  0.04
random_forest  0.15  0.22


In [13]:
print("Tree scores - unique values:", len(np.unique(tree_scores)))
print("Forest scores - unique values:", len(np.unique(forest_scores)))
print("\nAre tree and forest scores identical?", np.allclose(tree_scores, forest_scores))

print("\nFirst 10 tree scores:", tree_scores[:10])
print("First 10 forest scores:", forest_scores[:10])

Tree scores - unique values: 12
Forest scores - unique values: 167

Are tree and forest scores identical? False

First 10 tree scores: [0.66907015 0.49479584 0.49479584 0.49479584 0.49479584 0.36127776
 0.68673856 0.49479584 0.49479584 0.49479584]
First 10 forest scores: [0.555 0.21  0.155 0.305 0.225 0.045 0.445 0.07  0.35  0.265]


In [14]:
print("y_train decline rate:", y_train.mean())
print("y_test decline rate:", y_test.mean())
print("X_train describe:")
print(X_train.describe())

y_train decline rate: 0.1537572692314978
y_test decline rate: 0.17260469684793286
X_train describe:
        word_count  search_volume  feb_impressions       feb_ctr  \
count      52791.0        52791.0     52791.000000  52791.000000   
mean   3189.339149      70.610521      2177.830047      0.003695   
std    1254.571757     564.971023      5447.048008      0.021916   
min            0.0            0.0         1.000000      0.000000   
25%         2494.0            0.0        44.000000      0.000000   
50%         2857.0            0.0       446.000000      0.000607   
75%         3626.0           20.0      2252.000000      0.003401   
max        29341.0        40500.0    167303.000000      1.000000   

       feb_avg_position  
count      52791.000000  
mean          11.599522  
std           11.565725  
min            0.125000  
25%            5.176348  
50%            7.871291  
75%           13.783131  
max          285.000000  


In [15]:
tree_no_weight = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_no_weight.fit(X_train, y_train)
tree_scores_v2 = tree_no_weight.predict_proba(X_test)[:, 1]

print("Precision@20 (tree, no class_weight):", precision_at_k(tree_scores_v2, y_test_arr, 20))
print("Precision@50 (tree, no class_weight):", precision_at_k(tree_scores_v2, y_test_arr, 50))

Precision@20 (tree, no class_weight): 0.1
Precision@50 (tree, no class_weight): 0.16


In [16]:
print("Are baseline and forest scores identical?",
      np.allclose(test_table["baseline_score"].values, forest_scores))

Are baseline and forest scores identical? False


# ============================================================
# SECTION 3 SUMMARY: What we did, what went wrong, how we fixed it
# ============================================================
#
# GOAL: Compare our w04 baseline rule against two trained models
# (Decision Tree, Random Forest), on the SAME held-out test set
# (client-grouped split from Section 2), using the SAME metric
# (Precision@K, K=20 and K=50) we've used throughout this project.
#
# ATTEMPT 1: Trained both models normally, tree used class_weight="balanced".
# Result:
#   baseline_rule:  P@20=0.15, P@50=0.22
#   decision_tree:  P@20=0.05, P@50=0.04   <- worse than baseline, surprising
#   random_forest:  P@20=0.15, P@50=0.22   <- exactly matched baseline
#
# INVESTIGATION:
# 1. Checked whether tree and forest scores were literally identical (a bug)
#    -> np.allclose() returned False. Not a bug -- genuinely different scores.
# 2. Checked whether class_weight="balanced" was distorting the tree's
#    probability ranking (a known risk with shallow trees + imbalanced data)
#    -> Retrained WITHOUT class_weight="balanced":
#       decision_tree (fixed): P@20=0.10, P@50=0.16
#    -> Real improvement, confirming class_weight was hurting the tree's
#       ranking specifically (not just a random fluke).
# 3. Checked whether baseline_rule and random_forest scores were literally
#    identical (another possible bug)
#    -> np.allclose() returned False. Not a bug -- the RAW scores differ;
#       they only matched after being ROUNDED to 2 decimal places for the
#       table. Since Precision@20 can only take values in steps of 1/20
#       (0.05, 0.10, 0.15...) and Precision@50 in steps of 1/50, two
#       genuinely different rankings landing on the same rounded value by
#       coincidence is not unusual at these small K values.
#
# CONCLUSION (honest, final numbers):
#   baseline_rule:            P@20=0.15, P@50=0.22
#   decision_tree (corrected): P@20=0.10, P@50=0.16   <- below baseline
#   random_forest:             P@20=0.15, P@50=0.22   <- ties baseline (coincidence of rounding, confirmed different underlying scores)
#
# TAKEAWAY: Our simple hand-written baseline rule is a genuinely strong,
# hard-to-beat comparison point for this problem. Neither trained model
# clearly outperforms it with just our 5 current features -- worth
# investigating further in error analysis (Section 4) rather than assuming
# "more complex = automatically better."
# ============================================================

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [17]:
importances = pd.DataFrame({
    "feature": feature_cols,
    "tree_importance": tree_no_weight.feature_importances_,
    "forest_importance": forest.feature_importances_
}).sort_values("forest_importance", ascending=False)

print(importances)

            feature  tree_importance  forest_importance
0        word_count         0.468856           0.296447
4  feb_avg_position         0.085389           0.277160
2   feb_impressions         0.165546           0.245957
3           feb_ctr         0.161475           0.113888
1     search_volume         0.118734           0.066548


## Errors and interpretation

**What the models lean on:** Both the decision tree and random forest gave `word_count` the
highest importance (0.47 and 0.30 respectively) — notably higher than `feb_ctr` (0.16 and
0.11) or `feb_avg_position` (0.09 and 0.28), even though CTR-vs-position was our one
CONFIRMED, verified signal from w04's signal audit, while word_count was never checked at all.

**This is a real tension worth naming honestly:** high feature importance does not guarantee
a feature is genuinely predictive on new data — it just means the model leaned on it heavily
while fitting the training set. Our actual precision results support this caution: despite
word_count dominating importance, neither model clearly beat our simple baseline rule (which
uses only CTR and position, deliberately excluding word_count). The decision tree, in
particular, performed worse (P@20=0.10, P@50=0.16) than the baseline (P@20=0.15, P@50=0.22) —
suggesting that leaning heavily on an unverified signal like word_count did not translate into
better real-world ranking, and may reflect the model fitting patterns in the training data
that don't generalize as well to the held-out clients.

**Where the models are likely wrong:** Given random forest tied the baseline only because of
matched rounding at small K (confirmed the underlying scores differ), and the tree
underperformed even after fixing the class_weight issue — both models likely make different
kinds of errors than the baseline. The baseline is a strict, interpretable gate (must satisfy
BOTH position and CTR conditions); the tree/forest instead spread their attention across all 5
features, including one (word_count) we never verified as a genuine signal, which may explain
why they don't clearly outperform a simpler, more targeted rule.

**Takeaway:** With only 5 features and one verified signal, our simple, evidence-backed
baseline rule remains a strong, defensible choice — not automatically beaten by more complex
methods. This doesn't mean tree-based models are wrong for this problem in general; it
suggests we may need to either verify word_count as a real signal (a proper signal check,
like our w04 Signal 1/2 audits) before trusting it, or add richer features (e.g. query-
diversity signals mentioned in the reference notebook) before a model can meaningfully beat a
well-targeted hand rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.